In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the training and testing datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training dataset
print("Training Data Sample:")
print(train_df.head())

# Display the first few rows of the testing dataset
print("\nTesting Data Sample:")
print(test_df.head())

# Display the summary statistics of the training dataset
print("\nTraining Data Summary Statistics:")
print(train_df.describe())

# Display the summary statistics of the testing dataset
print("\nTesting Data Summary Statistics:")
print(test_df.describe())

# Display the data types of the training dataset
print("\nTraining Data Types:")
print(train_df.dtypes)

# Display the data types of the testing dataset
print("\nTesting Data Types:")
print(test_df.dtypes)

# Distinguish column types for tailored analysis and visualization
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

print("\nNumeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the correlation matrix for numeric columns
plt.figure(figsize=(12, 8))
correlation_matrix = train_df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=train_df)
plt.title('Distribution of Target Variable (Exited)')
plt.show()

# Visualize the distribution of categorical features
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(x=col, data=train_df)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.show()


Training Data Sample:
       id  CustomerId     Surname  ...  IsActiveMember EstimatedSalary Exited
0  149380    15780088  Yobachukwu  ...             1.0       103560.98      0
1  164766    15679760    Slattery  ...             0.0       102950.79      0
2  155569    15637678          Ma  ...             0.0       155394.52      0
3  124304    15728693      Galkin  ...             1.0       107428.42      0
4  108008    15613673        Lung  ...             0.0       134110.93      0

[5 rows x 14 columns]

Testing Data Sample:
       id  CustomerId       Surname  ...  IsActiveMember EstimatedSalary Exited
0   33042    15752375  Chukwumaobim  ...             0.0        79577.48      0
1   36330    15742681         P'eng  ...             0.0        38190.78      0
2   59446    15730397         Pinto  ...             0.0        69052.63      1
3   92278    15803365        Coffee  ...             0.0        62347.71      0
4  146750    15735270       Horsley  ...             0.0        9

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-15 07:02:49.777 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Surname', 'Geography', 'Gender'], 'Numeric': ['id', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Copy the DataFrames to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Handle missing values
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Surname', 'Geography', 'Gender']

# Fill missing values for numeric features with the mean
fill_missing_numeric = FillMissingValue(features=numeric_features, strategy='mean')
train_df_copy = fill_missing_numeric.fit_transform(train_df_copy)
test_df_copy = fill_missing_numeric.transform(test_df_copy)

# Fill missing values for categorical features with the most frequent value
fill_missing_categorical = FillMissingValue(features=categorical_features, strategy='most_frequent')
train_df_copy = fill_missing_categorical.fit_transform(train_df_copy)
test_df_copy = fill_missing_categorical.transform(test_df_copy)

# Encode categorical features using label encoding
label_encode = LabelEncode(features=categorical_features)
train_df_copy = label_encode.fit_transform(train_df_copy)
test_df_copy = label_encode.transform(test_df_copy)

# Standardize numeric features
standard_scale = StandardScale(features=numeric_features)
train_df_copy = standard_scale.fit_transform(train_df_copy)
test_df_copy = standard_scale.transform(test_df_copy)

# Display the first few rows of the preprocessed data
print("Preprocessed Training Data Sample:")
print(train_df_copy.head())
print("\nPreprocessed Testing Data Sample:")
print(test_df_copy.head())


Preprocessed Training Data Sample:
       id  CustomerId  Surname  ...  IsActiveMember  EstimatedSalary  Exited
0  149380    15780088     2703  ...        1.005415        -0.181454       0
1  164766    15679760     2302  ...       -0.994614        -0.193591       0
2  155569    15637678     1510  ...       -0.994614         0.849538       0
3  124304    15728693      888  ...        1.005415        -0.104529       0
4  108008    15613673     1505  ...       -0.994614         0.426198       0

[5 rows x 14 columns]

Preprocessed Testing Data Sample:
       id  CustomerId  Surname  ...  IsActiveMember  EstimatedSalary  Exited
0   33042    15752375      519  ...       -0.994614        -0.658497       0
1   36330    15742681     1949  ...       -0.994614        -1.481697       0
2   59446    15730397     2026  ...       -0.994614        -0.867841       1
3   92278    15803365      546  ...       -0.994614        -1.001205       0
4  146750    15735270     1130  ...       -0.994614        -

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

# Assuming train_df_copy and test_df_copy are already preprocessed
X_train = train_df_copy.drop(columns=['Exited'])
y_train = train_df_copy['Exited']
X_test = test_df_copy.drop(columns=['Exited'])
y_test = test_df_copy['Exited']

# Initialize and train the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Calculate and display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Calculate and display the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC Score: {auc_roc:.4f}")

# Display the classification report
class_report = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(class_report)

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'CustomerId': test_df['CustomerId'], 'Exited': y_pred})
predictions_df.to_csv('customer_churn_predictions.csv', index=False)
print("\nPredictions saved to 'customer_churn_predictions.csv'")


Confusion Matrix:
[[25063   989]
 [ 3507  3448]]

AUC-ROC Score: 0.8881

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92     26052
           1       0.78      0.50      0.61      6955

    accuracy                           0.86     33007
   macro avg       0.83      0.73      0.76     33007
weighted avg       0.86      0.86      0.85     33007


Predictions saved to 'customer_churn_predictions.csv'


In [6]:
# Evaluate the trained Random Forest model on the preprocessed testing dataset, generate predictions, and save the results.

# Ensure the evaluated data is same processed as the training data
X_test = test_df_copy.drop(columns=['Exited'])
y_test = test_df_copy['Exited']

# Use trained model from previous task result directly
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Generate and display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Calculate and display the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC Score: {auc_roc:.4f}")

# Generate and display the classification report
class_report = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(class_report)

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'CustomerId': test_df['CustomerId'], 'Exited': y_pred})
predictions_df.to_csv('customer_churn_predictions.csv', index=False)
print("\nPredictions saved to 'customer_churn_predictions.csv'")


Confusion Matrix:
[[25063   989]
 [ 3507  3448]]

AUC-ROC Score: 0.8881

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92     26052
           1       0.78      0.50      0.61      6955

    accuracy                           0.86     33007
   macro avg       0.83      0.73      0.76     33007
weighted avg       0.86      0.86      0.85     33007


Predictions saved to 'customer_churn_predictions.csv'


In [7]:
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load the preprocessed test data
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/test.csv'
test_df = pd.read_csv(test_path)

# Assuming the preprocessing steps are the same as before
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Surname', 'Geography', 'Gender']

# Apply the same preprocessing steps as before
fill_missing_numeric = FillMissingValue(features=numeric_features, strategy='mean')
test_df = fill_missing_numeric.transform(test_df)
fill_missing_categorical = FillMissingValue(features=categorical_features, strategy='most_frequent')
test_df = fill_missing_categorical.transform(test_df)
test_df = label_encode.transform(test_df)
test_df = standard_scale.transform(test_df)

# Prepare the test data for prediction
X_test = test_df.drop(columns=['Exited'])
y_test = test_df['Exited']

# Predict using the trained model
y_pred = rf_model.predict(X_test)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


NotFittedError: This SimpleImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.